# Improved 10-K AI focus processing

In [1]:

import os
import re
import html
from pathlib import Path
from collections import Counter

import pandas as pd
from tqdm import tqdm

# =========================
# USER SETTINGS
# =========================
DICT_FILE = r"/Volumes/ORICO/Dissertation/output/Hospitality_Full_Tech_Dictionary_v1_MERGED.xlsx"   # or your dictionary file
DICT_SHEET = "Dictionary"  # sheet containing the final terms
TEXT_FOLDER = r"/Volumes/ORICO/Dissertation/10-K"
OUTPUT_FILE = r"/Volumes/ORICO/Dissertation/Full_tech_Focus_Results.xlsx"
LOG_FILE = r"/Volumes/ORICO/Dissertation/processed_files.txt"

USE_INCREMENTAL_LOG = False
SAVE_TERM_COUNTS = True           # save matched-term frequency summary
EXTRACT_KEY_SECTIONS_ONLY = False # set True later if you want Item 1 / 1A / 7 only


# =========================
# LOAD DICTIONARY
# =========================
import pandas as pd

def load_terms(dict_file, sheet_name="Dictionary", term_col_candidates=("Terms", "Term", "Keyword", "Keywords")):
    """
    Load dictionary terms from Excel, even if the first row is a title row.
    Automatically finds the header row containing 'Terms'.
    """

    # 먼저 헤더 없이 읽어서 어느 행이 실제 헤더인지 찾기
    raw = pd.read_excel(dict_file, sheet_name=sheet_name, header=None)

    header_row = None
    for i in range(min(10, len(raw))):
        row_values = [str(x).strip().lower() for x in raw.iloc[i].tolist() if pd.notna(x)]
        if any(v in [c.lower() for c in term_col_candidates] for v in row_values):
            header_row = i
            break

    if header_row is None:
        raise ValueError(
            f"Could not find a header row with one of {term_col_candidates} "
            f"in the first 10 rows of sheet '{sheet_name}'."
        )

    # 찾은 헤더 행으로 다시 읽기
    df = pd.read_excel(dict_file, sheet_name=sheet_name, header=header_row)

    term_col = None
    for c in df.columns:
        if str(c).strip().lower() in [x.lower() for x in term_col_candidates]:
            term_col = c
            break

    if term_col is None:
        raise ValueError(f"Could not find a term column in {df.columns.tolist()}")

    terms = (
        df[term_col]
        .dropna()
        .astype(str)
        .str.strip()
        .tolist()
    )

    # 빈 문자열 제거 + 중복 제거
    terms = [t for t in terms if t]
    terms = list(dict.fromkeys(terms))

    return terms


def normalize_term(term: str) -> str:
    term = html.unescape(term)
    term = term.lower().strip()

    # standardize hyphen / slash / underscore variants into spaces
    term = term.replace("-", " ").replace("_", " ").replace("/", " ")

    # remove periods inside abbreviations like a.i. -> ai
    term = term.replace(".", "")

    # collapse whitespace
    term = re.sub(r"\s+", " ", term).strip()
    return term


# =========================
# TEXT CLEANING
# =========================
def read_text_file(file_path: Path) -> str:
    for enc in ("utf-8", "utf-8-sig", "latin-1", "cp1252"):
        try:
            return file_path.read_text(encoding=enc, errors="ignore")
        except Exception:
            continue
    raise ValueError(f"Could not read {file_path}")


def strip_non_text_exhibits(text: str) -> str:
    # Remove non-text SEC sections
    text = re.sub(r"(?is)<TYPE>\s*(GRAPHIC|ZIP|EXCEL|PDF|XML|JSON|JPEG|PNG).*?</DOCUMENT>", " ", text)
    return text


def strip_html_xbrl(text: str) -> str:
    text = html.unescape(text)

    # Remove script/style/comments
    text = re.sub(r"(?is)<script.*?>.*?</script>", " ", text)
    text = re.sub(r"(?is)<style.*?>.*?</style>", " ", text)
    text = re.sub(r"(?is)<!--.*?-->", " ", text)

    # Remove inline XBRL tags
    text = re.sub(r"(?is)</?(ix:|xbrli:|dei:|us-gaap:|xbrldi:)[^>]*>", " ", text)

    # Remove whole TABLE blocks first to cut noisy financial tables
    text = re.sub(r"(?is)<table.*?>.*?</table>", " ", text)

    # Remove all remaining tags
    text = re.sub(r"(?is)<[^>]+>", " ", text)

    return text


def normalize_text(text: str) -> str:
    text = text.replace("\xa0", " ").replace("\u200b", " ")

    # unify punctuation variants
    text = text.replace("–", "-").replace("—", "-").replace("’", "'")

    # remove periods in abbreviations / token noise: a.i. -> ai
    text = text.replace(".", "")

    # normalize separators so machine-learning == machine learning
    text = text.replace("-", " ").replace("_", " ").replace("/", " ")

    # remove SEC header/footer noise
    text = re.sub(r"(?i)table of contents", " ", text)
    text = re.sub(r"(?i)page\s+\d+", " ", text)
    text = re.sub(r"(?i)end privacy enhanced message", " ", text)

    # IMPORTANT: do NOT remove all uppercase words
    # They may contain useful signals such as AI, CRM, ERP, POS, etc.

    # remove possessive "'s" lightly
    text = re.sub(r"\b(\w+)'s\b", r"\1", text)

    # collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


def clean_text(raw_text: str) -> str:
    text = strip_non_text_exhibits(raw_text)
    text = strip_html_xbrl(text)
    text = normalize_text(text)
    return text


# =========================
# OPTIONAL SECTION EXTRACTION
# =========================
def extract_key_sections(text: str) -> str:
    specs = [
        (
            r"(?im)\bitem\s+1\s*[:\.-]?\s*business\b",
            [
                r"(?im)\bitem\s+1a\s*[:\.-]?\s*risk\s+factors\b",
                r"(?im)\bitem\s+2\b",
            ],
        ),
        (
            r"(?im)\bitem\s+1a\s*[:\.-]?\s*risk\s+factors\b",
            [
                r"(?im)\bitem\s+1b\b",
                r"(?im)\bitem\s+2\b",
            ],
        ),
        (
            r"(?im)\bitem\s+7\s*[:\.-]?\s*(management'?s\s+discussion\s+and\s+analysis|md&a)\b",
            [
                r"(?im)\bitem\s+7a\b",
                r"(?im)\bitem\s+8\b",
            ],
        ),
    ]

    chunks = []
    for start_pat, end_pats in specs:
        m = re.search(start_pat, text)
        if not m:
            continue
        start = m.start()
        tail = text[m.end():]
        ends = []
        for ep in end_pats:
            em = re.search(ep, tail)
            if em:
                ends.append(m.end() + em.start())
        end = min(ends) if ends else len(text)
        chunks.append(text[start:end].strip())

    return "\n\n".join(chunks).strip() if chunks else text


# =========================
# MATCHING
# =========================
def build_term_pattern(terms):
    escaped = [re.escape(t) for t in terms]
    # Because hyphens / underscores / slashes were normalized to spaces,
    # matching with simple word boundaries now works more reliably.
    pattern = re.compile(r"\b(" + "|".join(escaped) + r")\b", flags=re.IGNORECASE)
    return pattern


def count_terms(cleaned_text: str, pattern: re.Pattern):
    matches = pattern.findall(cleaned_text)
    counts = Counter(m.lower() for m in matches)
    return counts


def word_count(text: str) -> int:
    return len(re.findall(r"\b[a-zA-Z][a-zA-Z0-9]*\b", text))


# =========================
# INCREMENTAL LOG
# =========================
def load_processed(log_file: str):
    if os.path.exists(log_file):
        with open(log_file, "r", encoding="utf-8") as f:
            return set(line.strip() for line in f if line.strip())
    return set()


def append_processed(log_file: str, filename: str):
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(filename + "\n")


# =========================
# MAIN
# =========================
terms = load_terms(DICT_FILE, sheet_name=DICT_SHEET)
pattern = build_term_pattern(terms)

folder = Path(TEXT_FOLDER)
file_list = sorted([p for p in folder.iterdir() if p.suffix.lower() == ".txt"])

processed = load_processed(LOG_FILE) if USE_INCREMENTAL_LOG else set()
remaining_files = [p for p in file_list if p.name not in processed]

print(f"Dictionary terms loaded: {len(terms)}")
print(f"Total txt files: {len(file_list)}")
print(f"Already processed: {len(processed)}")
print(f"Remaining: {len(remaining_files)}")

all_rows = []
term_detail_rows = []

for file_path in tqdm(remaining_files, desc="Processing 10-K files", unit="file"):
    try:
        raw_text = read_text_file(file_path)
        cleaned_text = clean_text(raw_text)

        if EXTRACT_KEY_SECTIONS_ONLY:
            cleaned_text = extract_key_sections(cleaned_text)

        total_words = word_count(cleaned_text)
        term_counts = count_terms(cleaned_text, pattern)
        ai_terms_count = sum(term_counts.values())
        ai_focus = (ai_terms_count / total_words) * 1_000_000 if total_words > 0 else 0.0

        all_rows.append({
            "Filename": file_path.name,
            "AI_Terms_Count": ai_terms_count,
            "Total_Word_Count": total_words,
            "AI_Focus_per_million_words": round(ai_focus, 4),
            "Unique_AI_Terms_Matched": len(term_counts),
        })

        if SAVE_TERM_COUNTS and term_counts:
            for term, cnt in term_counts.items():
                term_detail_rows.append({
                    "Filename": file_path.name,
                    "Matched_Term": term,
                    "Count": cnt,
                })

        if USE_INCREMENTAL_LOG:
            append_processed(LOG_FILE, file_path.name)

    except Exception as e:
        all_rows.append({
            "Filename": file_path.name,
            "AI_Terms_Count": None,
            "Total_Word_Count": None,
            "AI_Focus_per_million_words": None,
            "Unique_AI_Terms_Matched": None,
            "Error": str(e),
        })

# save
summary_df = pd.DataFrame(all_rows)
summary_df = summary_df.sort_values("Filename").reset_index(drop=True)

if SAVE_TERM_COUNTS:
    detail_df = pd.DataFrame(term_detail_rows)
else:
    detail_df = pd.DataFrame(columns=["Filename", "Matched_Term", "Count"])

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    summary_df.to_excel(writer, index=False, sheet_name="Summary")
    detail_df.to_excel(writer, index=False, sheet_name="Term_Detail")
    pd.DataFrame({"Term": terms}).to_excel(writer, index=False, sheet_name="Dictionary_Used")

print(f"Done. Results saved to: {OUTPUT_FILE}")


Dictionary terms loaded: 118
Total txt files: 1518
Already processed: 0
Remaining: 1518


Processing 10-K files: 100%|████████████| 1518/1518 [5:00:38<00:00, 11.88s/file]


Done. Results saved to: /Volumes/ORICO/Dissertation/Full_tech_Focus_Results.xlsx
